In [9]:
import numpy as np
import random
import math
import pandas as pd
import copy

In [10]:


def read_cities_information():
    file1='./city1.tsp'
    file2='./city2.tsp'
    file3='./city3.tsp'

    data=[]
    with open(file1,'r') as file:
        for line in file:
            temp=line.split()
            if len(temp)==3:
                data.append([float(temp[1]),float(temp[2])])
    map1=np.array(data)
    data=[]

    with open(file2,'r') as file:
        for line in file:
            temp=line.split()
            if len(temp)==3:
                data.append([float(temp[1]),float(temp[2])])
    map2=np.array(data)
    data=[]

    with open(file3,'r') as file:
        for line in file:
            temp=line.split()
            if len(temp)==3:
                data.append([float(temp[1]),float(temp[2])])
    map3=np.array(data)
    data=[]

    return map1,map2,map3


In [11]:
def make_initial_population(num_cities,num_indivituals):
    population=[]

    for i in range(num_indivituals):
        new_individual=list(range(num_cities))
        random.shuffle(new_individual)
        population.append(new_individual)
    
    return population


In [12]:
def fitness_function(map,individual):
    cost=0
    for i in range(len(individual)-1):
        cost+=np.linalg.norm(map[individual[i]]-map[individual[i+1]],ord=2)
    cost+=np.linalg.norm(map[individual[-1]]-map[individual[0]],ord=2)

    return 1/cost


In [13]:
def order_crossover(parent1,parent2):
    num_cities=len(parent1)
    i,j=random.sample(range(num_cities),2)
    if i>j:
        i,j=j,i
    child1,child2=[None]*num_cities,[None]*num_cities

    child1[i:j+1]=parent1[i:j+1]
    index=0
    for i in range(num_cities):
        if child1[i] is None:
            while parent2[index] in child1:
                index+=1
            child1[i]=parent2[index]
    
    child2[i:j+1]=parent2[i:j+1]
    index=0
    for i in range(num_cities):
        if child2[i] is None:
            while parent1[index] in child2:
                index+=1
            child2[i]=parent1[index]

    return child1,child2



def pmx_croosover(parent1,parent2):
    num_cities=len(parent1)
    i,j=random.sample(range(num_cities),2)
    if i>j:
        i,j=j,i
    child1,child2=[None]*num_cities,[None]*num_cities
    mapping=dict()

    for k in range(i,j+1):
        child1[k]=parent2[k]
        mapping[parent2[k]]=parent1[k]
    for k in range(num_cities):
        if child1[k] is None:
            val=parent1[k]
            while val in mapping:
                val=mapping[val]
            child1[k]=val
    
    
    for k in range(i,j+1):
        child2[k]=parent1[k]
        mapping[parent1[k]]=parent2[k]
    for k in range(num_cities):
        if child2[k] is None:
            val=parent2[k]
            while val in mapping:
                val=mapping[val]
            child2[k]=val

    return child1,child2


def cycle_croosover(parent1,parent2):
    num_cities=len(parent1)
    map1={i:j for j , i in enumerate(parent1)}
    visited=[False]*num_cities
    cycles=[]
    child1,child2=[None]*num_cities,[None]*num_cities

    for i in range(num_cities):
        if not visited[i]:
            cycle=[]
            current=i
            while not visited[current]:
                visited[current]=True
                cycle.append(current)
                current=map1[parent2[current]]
            
            cycles.append(cycle)
    
    for i , cycle in enumerate(cycles):
        if i%2==0:
            for j in cycle:
                child1[j]=parent1[j]
                child2[j]=parent2[j]
        else:
            for j in cycle:
                child1[j]=parent2[j]
                child2[j]=parent1[j]

    return child1,child2

def position_based_crossover(parent1,parent2):
    num_cities=len(parent1)
    child1,child2=[None]*num_cities,[None]*num_cities
    positions=random.sample(range(num_cities),num_cities//2)

    for position in positions:
        child1[position]=parent1[position]
        child2[position]=parent2[position]

    index=0
    for i in range(num_cities):
        if child1[i] is None:
            while parent2[i] in child1:
                index+=1
            child1[i]=parent2[index]

    index=0
    for i in range(num_cities):
        if child2[i] is None:
            while parent1[i] in child2:
                index+=1
            child2[i]=parent1[index]

    return child1,child2

    

In [14]:
def two_swap_mutation(individual):
    num_cities=len(individual)
    i,j=random.sample(range(num_cities),2)
    individual[i],individual[j]=individual[j],individual[i]
    return individual

def inversion_mutation(individual):
    num_cities=len(individual)
    i,j=random.sample(range(num_cities),2)
    if i>j:
        i,j=j,i
    individual[i:j+1]=individual[i:j+1][::-1]
    return individual

def scramble_mutation(individual):
    num_cities=len(individual)
    i,j=random.sample(range(num_cities),2)
    if i>j:
        i,j=j,i
    temp=individual[i:j+1]
    random.shuffle(temp)
    individual[i:j+1]=temp
    return individual

def displacement_mutation(individual):
    num_cities=len(individual)
    i,j=random.sample(range(num_cities),2)
    if i>j:
        i,j=j,i
    temp=individual[i:j+1]
    del individual[i:j+1]
    new_pos=random.randint(range(len(individual)))
    individual[new_pos:new_pos]=temp
    return individual


In [15]:
def k_tournament(k,population_size,population,map):
    best_fitness=-1
    winner=[]

    for i in range(k):
        r=random.randint(0,population_size-1)
        fitness=fitness_function(map,population[r])
        if i == 0 :
            best_fitness=fitness
            winner=population[r]
        if fitness>best_fitness:
            best_fitness=fitness
            winner=population[r]

    return winner

In [ ]:
map1,map2,map3=read_cities_information()

# GA 
crossovers=[order_crossover,pmx_croosover,cycle_croosover,position_based_crossover]
mutations=[two_swap_mutation,inversion_mutation,scramble_mutation,displacement_mutation]
num_iterations=int(input('Enter number of iterations: '))
num_population=int(input('Enter number of individuals: '))
k=int(input('Enrer k parameter of tournament selection: '))
p=float(input('Enter the probability of mutation: '))
map_index=int(input('Enter desired map number (1/2/3): '))
crossover=int(input('Enter desired crossover number (1.Order crossover, 2.PMX crossover, 3.Cycle crossover, 4.Position Based crossover):'))
mutation=int(input('Enter desired mutation number (1.2-Swap mutation, 2.Inversion mutation, 3.Scramble mutation, 4.Displacement mutation):'))
my_map=0
if map_index == 1:
    my_map=map1
elif map_index == 2:
    my_map=map2
else:
    my_map=map3

population=make_initial_population(len(my_map),num_population)

best_fitness,best_solution=-1,0

for i in range(num_iterations):
    children=[]
    for j in range(num_population//2):
        parent1=k_tournament(k,num_population,population,my_map)
        parent2=k_tournament(k,num_population,population,my_map)

        child1,child2=crossovers[crossover-1](parent1,parent2)
        if random.random()<p:
            child1,child2=mutations[mutation-1](child1),mutations[mutation-1](child2)

        fitness1,fitness2=fitness_function(my_map,child1),fitness_function(my_map,child2)

        if fitness1>best_fitness:
            best_fitness=fitness1
            best_solution=child1
        
        if fitness2>best_fitness:
            best_fitness=fitness2
            best_solution=child2

        
        children.append(child1)
        children.append(child2)

    temp=population+children
    random.shuffle(temp)
    population=list(random.sample(temp,num_population))


print(f'The length of best computed path for map number {map_index} is ',1/best_fitness)
    
    
    



